In [ ]:
import geopandas as gpd
import pandas as pd
import networkx as nx
import h3
import igraph as ig
import leidenalg
from tqdm import tqdm
from sklearn.preprocessing import normalize
import numpy as np
import random
seed = 505
random.seed(seed)        # or any integer
np.random.seed(seed)

In [ ]:
# Load data
input_dir = "../data/output_files/retailcen_foursquare_mrc_2_mri_10_mrn_10_mtf_6_merge_n/"
retail_centres = gpd.read_parquet(input_dir + "3_afterfill.parquet")


# Normalize POI vectors to unit length (row-wise L2 norm)
poi_cols = ['Dining and Drinking' , 'Leisure & Travel', 'Community & Health','Commerce']
retail_centres[poi_cols] = retail_centres[poi_cols].fillna(0)
poi_matrix = normalize(retail_centres[poi_cols].values, norm='l2', axis=1)
retail_centres['poi_vec'] = poi_matrix.tolist()

In [88]:
alphas = [0,0.5,1]  # Density to POI weighting
# resolutions = [0.01, 0.05, 0.08, 0.1, 0.12, 0.15, 0.3]
resolutions = [0.01,0.05,0.1,0.12,0.15,0.2,0.3,0.5]  # Resolution parameters for clustering
all_clustered = []

# Renumber networks by size (largest gets 1, next 2, ...)
retail_centres['network_number'] = retail_centres['network_number'].map(retail_centres['network_number'].value_counts().rank(method='first', ascending=False).astype(int))
network_numbers = retail_centres['network_number'].unique()

for network_number in tqdm(network_numbers):
    rc = retail_centres[retail_centres['network_number'] == network_number].copy()

    if len(rc) < 2:
        # Not enough to cluster
        for alpha in alphas:
            for res in resolutions:
                rc[f'subcluster_leiden_r{res}_a{alpha}'] = 0
        all_clustered.append(rc)
        continue

    # Build lookup dictionaries once
    poi_lookup = dict(zip(rc['h3_11'], rc['poi_vec']))
    density_lookup = dict(zip(rc['h3_11'], rc['foursquare_count']))
    h3_set = set(rc['h3_11'])

    for alpha in alphas:
        # Build graph
        G = nx.Graph()
        # print(f"Building graph for network number {network_number} with alpha {alpha}")
        for h in sorted(h3_set):
            G.add_node(h, density=density_lookup[h])
            neighbors = h3.grid_ring(h, 1)
            for n in sorted(neighbors):
                if n in h3_set and not G.has_edge(h, n):
                    sim = np.dot(poi_lookup[h], poi_lookup[n])
                    if sim == 0:
                        sim = 0.025  # Set a minimum similarity threshold so weights are not zero in case of aplha = 1
                    d1, d2 = density_lookup[h], density_lookup[n]
                    # density_weight = 1 / (1 + abs(d1 - d2))
                    density_weight = d1 + d2 + 1  # Use sum of densities as weight
                    weight = alpha * sim + (1 - alpha) * density_weight

                    if weight > 0:
                        G.add_edge(h, n, weight=weight)
        # Prepare igraph
        h3_nodes = list(G.nodes)
        h3_id_map = {h: i for i, h in enumerate(h3_nodes)}
        edges = [(h3_id_map[u], h3_id_map[v]) for u, v in G.edges()]
        weights = [G[u][v]['weight'] for u, v in G.edges()]

        if edges:
            G_ig = ig.Graph(n=len(h3_nodes), edges=edges, directed=False)

            G_ig.es['weight'] = weights
        else:
            G_ig = None

        for res in resolutions:
            if G_ig:
                partition_leiden = leidenalg.find_partition(
                    G_ig,
                    leidenalg.RBConfigurationVertexPartition,
                    weights=G_ig.es['weight'],
                    resolution_parameter=res,
                    seed=seed,
                    n_iterations=-1
                )
                membership = partition_leiden.membership
                h3_to_leiden = {h3_nodes[i]: membership[i] for i in range(len(h3_nodes))}
                rc[f'subcluster_leiden_r{res}_a{alpha}'] = rc['h3_11'].map(h3_to_leiden)
            else:
                rc[f'subcluster_leiden_r{res}_a{alpha}'] = 0

    all_clustered.append(rc)

# Combine all results
retail_centres_clustered = pd.concat(all_clustered, ignore_index=True)
retail_centres_clustered.to_parquet(input_dir + "4_subclustered_multi_resolution_alphas_sumweight.parquet", index=False)


  0%|          | 0/12044 [00:00<?, ?it/s]

100%|██████████| 12044/12044 [05:25<00:00, 37.02it/s]
